In [ ]:
# Load and compare mockTalents and talentsSeed
def read_file(path):
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

mock_data = read_file('src/data/mockData.js')
seed_data = read_file('camertalents-backend/seed.js')

mock_lines = mock_data.splitlines()
seed_lines = seed_data.splitlines()

# Helper to parse objects from an array definition by name
import re

def parse_js_array_objects(source, array_name):
    start = source.find(f'export const {array_name} = [')
    if start == -1:
        raise ValueError(f'{array_name} not found')
    body = source[start:]
    brace_level = 0
    objects = []
    current = []
    in_object = False
    for line in body.splitlines():
        if line.strip().startswith('export const'):
            continue
        if '{' in line and not in_object:
            in_object = True
        if in_object:
            current.append(line)
            brace_level += line.count('{') - line.count('}')
            if in_object and brace_level == 0 and current:
                objects.append('\n'.join(current))
                current = []
                in_object = False
    return objects

mock_objs = parse_js_array_objects(mock_data, 'mockTalents')
seed_objs = parse_js_array_objects(seed_data, 'talentsSeed')

# Extract names and dateInscription values
mock_map = {}
for obj in mock_objs:
    m = re.search(r"nom:\s*['\"]([^'\"]+)['\"]", obj)
    d = re.search(r"dateInscription:\s*['\"]([^'\"]+)['\"]", obj)
    if m and d:
        mock_map[m.group(1)] = d.group(1)

missing = []
for obj in seed_objs:
    m = re.search(r"nom:\s*['\"]([^'\"]+)['\"]", obj)
    if not m:
        continue
    name = m.group(1)
    if 'dateInscription:' not in obj and name in mock_map:
        missing.append((name, mock_map[name]))

print('Found', len(seed_objs), 'seed entries and', len(mock_objs), 'mock entries')
print('Missing dateInscription count in seed:', len(missing))
for name, date in missing:
    print(name, '->', date)
